# 02 — Simple RAG + Personas + Chat Memory

**Big idea:** retrieve relevant context first, then generate with a clear system prompt.

## Learning goals

By the end, you can explain:
- what retrieval-augmented generation (RAG) does
- how system prompts change assistant behavior
- how chat history is passed back to the model each turn

## Workflow map

1. Build a tiny local knowledge base
2. Retrieve the most relevant chunk(s) for a question
3. Add retrieved context into the prompt
4. Generate answer with persona/system prompt
5. Inspect chat session memory

In [ ]:
# Install once if needed:
# !pip install -q gpt4all numpy

# Import GPT4All for local LLM chat and Embed4All for text embeddings
# numpy for vector operations (similarity calculation)
import numpy as np
from gpt4all import GPT4All, Embed4All

In [ ]:
# List all models available locally
# Format: model_name, filename (GGUF format)
for model in GPT4All.list_models():
    print(f"{model['name']}, {model['filename']}")

In [ ]:
# Load the model from disk (quantized GGUF format)

# Change MODEL_FILENAME to match a model from the list above
MODEL_FILENAME = "Nous-Hermes-2-Mistral-7B-DPO.Q4_0.gguf"
model = GPT4All(MODEL_FILENAME)

# Load the embedder (for later use in RAG)
embedder = Embed4All()
print("Model + embedder ready")

## Step 0 — Simplest RAG (5 lines of logic)

Before we build embeddings and retrieval functions, see how RAG works in its most basic form.

This step shows: **No embeddings, no fancy retrieval — just context + question → answer**

## Understanding RAG: The 3-Step Workflow

**RAG** = Retrieval-Augmented Generation

A better way to use LLMs: ground them in your facts instead of guessing.

### The 3 Core Steps:

1. **RETRIEVAL** — Find relevant info from your private data
   - Example: Search a PDF or knowledge base for chunks matching the user's question
   - Goal: Surface the facts the model didn't already know

2. **AUGMENTATION** — Stuff that info into the prompt as context
   - Example: "Here are the facts. Now answer based on them."
   - Goal: Force the model to answer from YOUR data, not its training

3. **GENERATION** — Let the LLM answer using that context
   - Example: The model reads your facts and extracts the answer
   - Goal: Reduce hallucination because the answer comes from your facts

**Why RAG matters:** Without retrieval, the model guesses. With it, the model reads.

In [ ]:
# === STEP 0: SIMPLE RAG ===

# Demonstrate the 3-step RAG workflow in the most minimal way

# 1. RETRIEVAL (manual for now—just hardcode the facts we want to use)
retrieved_context = "The Green Campus meeting is Friday 3:00 PM in Room 402. Pizza will be served."

# 2. AUGMENTATION (stuff context into prompt)
simple_question = "When is the meeting and what food is served?"
simple_rag_prompt = f"""
Use only this context to answer the question.
Context: {retrieved_context}
Question: {simple_question}
Answer:
""".strip()

# 3. GENERATION (let model answer based on context)
with model.chat_session(system_prompt="You are a helpful assistant. Answer only from context."):
    simple_answer = model.generate(simple_rag_prompt, max_tokens=60)

print("CONTEXT:", retrieved_context)
print("\nQUESTION:", simple_question)
print("\nANSWER:", simple_answer)

## Step 1 — Tiny local knowledge base (RETRIEVAL)

In a real system this would come from files (PDF/TXT). Here we keep it explicit for learning.

**This is the R in RAG:** We're preparing the facts to retrieve.

In [ ]:
# Create a tiny knowledge base (in production, this would come from documents)
chunks = [
    "The University Student Council meeting is scheduled for Friday at 3:00 PM in Room 402.",
    "The main topic is the new Green Campus initiative.",
    "Pizza will be served at the end of the meeting.",
    "All student clubs must submit budget requests by Wednesday 17:00.",
    "The design studio access card desk is open weekdays from 09:00 to 16:00."
]

# Embed all chunks into vectors for similarity comparison
chunk_embeddings = np.array([embedder.embed(c) for c in chunks])
print(f"Indexed {len(chunks)} chunks")

## Step 2 — Minimal retrieval function (RETRIEVAL)

This is a tiny in-memory retriever (no vector database).

**Still the R in RAG:** Now we're automatically finding the most relevant chunks for a query.

In [ ]:
# Calculate cosine similarity between two vectors
# Returns 1.0 for identical direction, 0.0 for perpendicular, -1.0 for opposite
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Retrieve top-k most similar chunks for a given query
def retrieve_top_k(query, k=2):
    # Embed the query
    q_emb = embedder.embed(query)
    # Calculate similarity scores against all chunks
    scores = [cosine_similarity(q_emb, emb) for emb in chunk_embeddings]
    # Sort by score (descending) and return top-k
    ranked = sorted(list(enumerate(scores)), key=lambda x: x[1], reverse=True)
    return [(chunks[i], s) for i, s in ranked[:k]]

In [ ]:
# Test the retriever: find relevant chunks for a user query
user_query = "When is the student council meeting and what food is served?"
retrieved = retrieve_top_k(user_query, k=2)

print("Query:", user_query)
print("\nTop retrieved chunks:")
# Display matched chunks with their similarity scores
for c, s in retrieved:
    print(f"- ({s:.3f}) {c}")

## Step 3 — Build a grounded RAG prompt (AUGMENTATION + GENERATION)

We instruct the model to answer from retrieved context only.

**The A in RAG:** We combine context + question into a prompt.
**The G in RAG:** The model generates an answer grounded in that context.

In [ ]:
# Build RAG prompt: combine retrieved context with the question
# This instructs the model to answer ONLY from provided context

# Join retrieved chunks into a single string
context_block = "\n".join([c for c, _ in retrieved])

# Build RAG prompt
rag_prompt = f"""
Use the provided Context only.
If the answer is not in Context, say: I don't know based on the provided context.

Context:
{context_block}

Question: {user_query}
Answer:
"""

# Stripping whitespace for cleaner formatting
rag_prompt = rag_prompt.strip()

# Generate answer with a system prompt defining the assistant role
with model.chat_session(system_prompt="You are a helpful university information assistant."):
    rag_answer = model.generate(rag_prompt, max_tokens=140)

print(rag_answer)

## Step 4 — Multiple personas on the same retrieved context (GENERATION VARIATIONS)

Same facts, different voice/behavior.

**Still the G:** We show how the system prompt changes voice without changing the facts (retrieval/augmentation stay the same).

In [ ]:
# Define multiple personas that will answer the same question differently
personas = {
    "tutor": "You are a supportive Academic Tutor. Explain clearly and encourage the student.",
    "admin": "You are a strict University Administrator. Give only direct facts in bullet points.",
    "coach": "You are an energetic student coach. Be concise and motivational."
}

# Generate answers for each persona using the same RAG context
for role, system_instruction in personas.items():
    print(f"\n--- PERSONA: {role.upper()} ---")
    with model.chat_session(system_prompt=system_instruction):
        answer = model.generate(rag_prompt, max_tokens=140)
    print(answer)

## Step 5 — Show chat memory explicitly (GENERATION WITH MEMORY)

Inside a chat session, previous turns are stored and sent back on next turns (until context window limit).

**Still the G:** Generation improved by remembering context across multiple questions.

In [ ]:
# Demonstrate chat memory: the model remembers previous turns within a session
memory_system_prompt = "You are the University Information Bot. Keep answers concise."

with model.chat_session(system_prompt=memory_system_prompt):

    # First turn: ask about the meeting
    q1 = "When is the meeting?"
    p1 = f"Context: {context_block}\n\nQuestion: {q1}"
    a1 = model.generate(p1, max_tokens=80)

    # Second turn: follow-up question
    q2 = "And what food is served?"
    p2 = f"Context: {context_block}\n\nQuestion: {q2}"
    a2 = model.generate(p2, max_tokens=80)

    # Access the chat session memory to see all stored turns
    session = model.current_chat_session

# Display results
print("Q1:", q1)
print("A1:", a1)
print()
print("Q2:", q2)
print("A2:", a2)
print()
print("Stored chat turns:", len(session))
# Show what the model stored in memory (first 220 chars per turn)
for turn in session:
    role = turn.get("role", "unknown")
    content = turn.get("content", "")
    print(f"\n[{role}]")
    print(content[:220])

## Reflection task

Write 4–6 lines:
- What part is retrieval, what part is generation?
- How did persona change outputs without changing facts?
- Why does inspecting `current_chat_session` matter for debugging?